# One EnergyPlus input → 5R1C and BRCM

This notebook demonstrates the shared conversion workflow. The EnergyPlus IDF is parsed once, physical quantities are normalized once, and that common data is passed to both `generate_5R1C()` and `generate_BRCM()`.

**The original `RC_br/ETHlib` package is not edited or monkey-patched.** The word *replacement* below means that this adapter currently supplies a different data source or calculation in place of behavior performed inside ETHlib's annual simulation loop.

## Data-source precedence

1. EnergyPlus geometry and fabric are the primary source.
2. BRCM's normalized conversion resolves `autocalculate` geometry.
3. JSON remains an explicit overlay for assumptions not reliably available from the current IDF conversion.
4. No weather, gain, control, or HVAC signal is silently invented.

In [ ]:
from pathlib import Path
import sys

ROOT = next(parent for parent in (Path.cwd(), *Path.cwd().parents) if (parent / 'src' / 'brcm').is_dir())
sys.path[:0] = [str(ROOT / 'src'), str(ROOT)]

from brcm_5r1c_workflow.workflow import generate_model_pair

HERE = ROOT / 'brcm_5r1c_workflow'
pair = generate_model_pair(
    ROOT / '_E+' / '1ZoneUncontrolled1.idf',
    idd_path=ROOT / '_E+' / 'idd' / '23.2' / 'Energy+.idd',
    defaults_json=HERE / 'config' / '5r1c_defaults.json',
)
pair.audit()

## ETHlib substitution and parity register

| Topic | Original ETHlib behavior | Current shared-workflow behavior | Status / action needed |
| --- | --- | --- | --- |
| Package code | `ETHlib` classes and annual loop calculate the model directly. | No ETHlib source file is overwritten. A separate adapter reproduces its 5R1C parameter definitions. | **Tracked; original preserved.** |
| Floor area and volume | Read from `geo_params.json`. | Derived from EnergyPlus zones/surfaces after BRCM resolves `autocalculate`; optional geometry JSON can still override. | **Intentional replacement.** Compare geometry before accepting an override. |
| Opaque envelope area | `WALL_AREA` comes from JSON. | Sum of outdoor surface areas minus hosted windows. | **Intentional replacement.** Roof is included in the exterior opaque aggregate; ETHlib's name `WALL_AREA` can obscure this distinction. |
| Opaque U/UA | `u_walls × WALL_AREA` from JSON. | Layer resistances from EnergyPlus materials/constructions are aggregated into UA. Current calculation excludes inside/outside film resistances. | **Partial parity.** Decide whether standard film resistances or EnergyPlus effective U-values should be used. |
| Window U/UA | `u_windows × WINDOW_AREA` from JSON. | Window area comes from EnergyPlus; corrected U-value remains a JSON overlay (`3.798 W/m²K` for this example) because the core converter exports a placeholder. | **Explicit override.** Later read the effective assembly value from EnergyPlus EIO/SQL. |
| Window SHGC / solar transmittance | ETHlib creates one south-facing window using `_beta`, transmittance `0.3`, and its radiation module. | No solar forcing is applied yet. Window geometry is parsed, but orientation-dependent irradiance and SHGC are not converted into a common signal. | **Not implemented—do not compare temperatures or energy yet.** |
| Solar radiation rate | ETHlib computes hourly solar gain internally from EPW direct/diffuse radiation and one assumed south window. | The shared model currently calculates no solar radiation rate and does not substitute zero during simulation because simulation is intentionally withheld. | **Required next:** common per-window incident/transmitted solar series, then aggregate the identical watts for both models. |
| Thermal mass | `thermal_capacitance_per_floor_area × FLOOR_AREA`, with fixed ISO mass area `2.5 × floor area`. | Volumetric material heat capacity is derived from EnergyPlus layer thickness, density, heat capacity and area. ISO mass area remains `2.5 × floor area` for 5R1C conductance. JSON capacitance is only a fallback. | **Intentional replacement with topology caveat.** BRCM distributes mass by layer; 5R1C lumps it into one node. |
| Total internal surface area | `_alpha × FLOOR_AREA` from JSON. | Same JSON convention is retained for `H_tr_is`; it is not yet derived from all inward-facing polygons. | **Preserved assumption.** Candidate for geometry-derived replacement. |
| Infiltration | ETHlib derives ACH from `infl_rate_m3ph_m2 × WALL_AREA / VOLUME`, plus atrium and window-opening ACH. | Supports explicit `ach_infl`; also supports the façade-rate term. Atrium and window-opening contributions are not automatically inferred from IDF schedules/objects. | **Partial parity.** Each omitted component must be explicitly supplied or reported as zero. |
| Mechanical ventilation | ETHlib derives design ACH from people × fresh-air rate, then changes it using a design occupancy schedule. | Supports explicit `ach_vent`, or derives design ACH when occupancy and fresh-air JSON inputs exist. No hourly controller has been applied. | **Static parameter only.** Hourly schedule/control adapter required. |
| Ventilation heat-loss load | ETHlib uses `1200 × b_ek × V × ACH / 3600`, where heat recovery modifies `b_ek`; its loop changes ACH and recovery by hour. | `H_ve,adj` uses the same formula but currently as one constant based on design ACH and one recovery efficiency. | **Formula preserved, dynamics missing.** Must recompute the coefficient hourly for both models. |
| Heat recovery | ETHlib enables recovery in selected winter months during designed occupied hours. | One JSON efficiency is used in the static audit. | **Schedule not reproduced.** Do not interpret annual load differences until aligned. |
| Schedules | ETHlib creates heating, occupancy, ventilation, and heat-recovery schedules in Python. | EnergyPlus schedule object names are inventoried, but schedule values are not expanded. | **Not implemented.** A common timestamped schedule table is required. |
| Internal gains | ETHlib combines people and appliance gains from occupancy and JSON rates. | No gain time series is currently generated or applied. | **Not implemented.** Define watts by zone and identical convective/radiant allocation. |
| Heating/cooling controls | ETHlib changes setpoints and solves ideal heating/cooling demand each hour. | Setpoints are present only as JSON metadata; neither model is controlled in this prototype. | **Not implemented.** One shared ideal-load controller is required. |
| Initial conditions | ETHlib initializes mass and air temperatures to `20°C`. | Models are generated but not simulated, so no initial temperature is imposed. | **Pending.** Use identical zone temperature and document how BRCM layer states are initialized. |
| Boundary conditions | ETHlib has one aggregated exterior temperature path. | BRCM retains ambient, adiabatic, ground and interzone surface distinctions; 5R1C aggregates exterior UA. | **Known topology difference.** Ground/interzone treatment must be mapped explicitly. |
| Energy and ECM savings | ETHlib reports heating/cooling energy and supports downstream ECM comparisons. | No energy or ECM result is produced yet. | **Correctly withheld** until forcing and control parity are demonstrated. |

## Current common quantities

This cell exposes the values actually passed into the two generators. It is an audit, not a claim of dynamic equivalence.

In [ ]:
{
    'floor_area_m2': pair.common.floor_area_m2,
    'volume_m3': pair.common.volume_m3,
    'opaque_envelope_area_m2': pair.common.opaque_envelope_area_m2,
    'window_area_m2': pair.common.window_area_m2,
    'opaque_UA_W_K': pair.common.opaque_ua_w_k,
    'window_UA_W_K': pair.common.window_ua_w_k,
    'thermal_capacity_J_K': pair.common.thermal_capacity_j_k,
    'infiltration_ACH': pair.common.infiltration_ach,
    'ventilation_ACH': pair.common.ventilation_ach,
    'boundaries': pair.common.boundary_counts,
    'BRCM_states': pair.brcm_model.state_identifiers,
}

## Gate before simulation comparison

A comparison may proceed only when a shared hourly forcing table supplies, with units and timestamps: outdoor and ground temperature; direct/diffuse solar and transmitted window solar; zone convective/radiant gains; ventilation and infiltration flow; heating/cooling setpoints; available HVAC power; and initial temperatures. The same table must feed both solvers without either solver independently rebuilding schedules or gains.

After that gate passes, report zone temperature RMSE/CVRMSE/NMBE, heating and cooling energy, calibration metrics against measurements, and baseline-versus-ECM savings for both topologies.